# Phase-1 EDA — TPUGraphs

Exploratory analysis built on the `src.data` pipeline. Renders per-collection counts, structural distributions, per-graph runtime ranges, the op-code vocabulary, `node_feat` column ranges (flagging negative / heavy-tailed columns), and a visual confirmation that **test labels are placeholders**. Figures are saved to `artifacts/figures/`. Hand back to the planner for review.

In [1]:
import os, sys
from pathlib import Path
# Run from the repo root whether launched there or from notebooks/.
_here = Path.cwd()
ROOT = _here if (_here / 'src').is_dir() else _here.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
from src.data.paths import COLLECTIONS, resolve_data_root, list_npz
from src.data.inventory import build_inventory
from src.data.cache import read_bundle
FIG = ROOT / 'artifacts' / 'figures'; FIG.mkdir(parents=True, exist_ok=True)
DATA = resolve_data_root(); print('data root:', DATA)

C:\Users\user\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


data root: data\npz\tpugraphs


## 1. Inventory & per-collection counts

In [2]:
inv_path = ROOT / 'artifacts' / 'inventory.parquet'
if inv_path.exists():
    inv = pd.read_parquet(inv_path)
else:
    inv = build_inventory(verbose=False); inv.to_parquet(inv_path, index=False)
counts = inv.groupby(['collection', 'split']).size().unstack(fill_value=0)
print(counts)
emb = int(inv.opcode_max.max()) + 1
print('global opcode_max =', int(inv.opcode_max.max()), '-> embedding size =', emb)
print('edge_oob =', int(inv.edge_oob.sum()),
      '| has_nan =', int(inv.has_nan.sum()), '| has_inf =', int(inv.has_inf.sum()))

split               test  train  valid
collection                            
layout:nlp:default    17    198     20
layout:nlp:random     17    207     20
layout:xla:default     8     61      7
layout:xla:random      8     69      7
tile:xla             844   5709    676
global opcode_max = 118 -> embedding size = 119
edge_oob = 0 | has_nan = 0 | has_inf = 0


In [3]:
ax = counts.plot(kind='bar', figsize=(9, 4), logy=True)
ax.set_ylabel('files (log)'); ax.set_title('Files per collection / split')
plt.tight_layout(); plt.savefig(FIG / '01_counts.png', dpi=140); plt.close()
print('saved', FIG / '01_counts.png')

saved C:\Users\user\.vscode\.vscode\NTU\SC4000\kaggle---prj\artifacts\figures\01_counts.png


## 2. Structural distributions (n_nodes, n_edges, n_configs, n_config_nodes)

In [4]:
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, col in zip(axes.ravel(), ['n_nodes', 'n_edges', 'n_configs', 'n_config_nodes']):
    for fam, sub in inv.groupby('family'):
        v = sub[col].dropna().astype(float); v = v[v > 0]
        if len(v):
            ax.hist(np.log10(v), bins=40, alpha=0.5, label=fam)
    ax.set_title(col); ax.set_xlabel('log10'); ax.set_ylabel('count'); ax.legend()
plt.tight_layout(); plt.savefig(FIG / '02_structure_hist.png', dpi=140); plt.close()
print('saved', FIG / '02_structure_hist.png')

saved C:\Users\user\.vscode\.vscode\NTU\SC4000\kaggle---prj\artifacts\figures\02_structure_hist.png


## 3. Per-graph runtime ranges (log scale)

Runtimes are comparable **only within a graph** — cross-graph scales are not. Each vertical bar is one graph's [min, max] runtime; the dot is the median.

In [5]:
rng = np.random.default_rng(0)
fig, ax = plt.subplots(figsize=(10, 4)); x0 = 0; ticks = []; labels = []
for coll in COLLECTIONS:
    files = [str(p) for p in list_npz(DATA, coll, 'train')]
    sel = rng.choice(len(files), size=min(15, len(files)), replace=False)
    start = x0
    for i in sel:
        rt = read_bundle(files[i])['config_runtime'].astype(float); rt = rt[rt > 0]
        if len(rt):
            ax.vlines(x0, rt.min(), rt.max(), color='steelblue')
            ax.plot(x0, np.median(rt), '.', color='darkorange'); x0 += 1
    ticks.append((start + x0) / 2); labels.append(coll.replace('layout:', '')); x0 += 2
ax.set_yscale('log'); ax.set_xticks(ticks); ax.set_xticklabels(labels, rotation=20)
ax.set_ylabel('config_runtime (log, ns)')
ax.set_title('Per-graph runtime range (sampled train graphs)')
plt.tight_layout(); plt.savefig(FIG / '03_runtime_ranges.png', dpi=140); plt.close()
print('saved', FIG / '03_runtime_ranges.png')

saved C:\Users\user\.vscode\.vscode\NTU\SC4000\kaggle---prj\artifacts\figures\03_runtime_ranges.png


## 4. Op-code histogram & vocabulary size

In [6]:
from collections import Counter
cnt = Counter()
for coll in COLLECTIONS:
    for f in [str(p) for p in list_npz(DATA, coll, 'train')[:30]]:
        with np.load(f) as z:
            cnt.update(z['node_opcode'].tolist())
ks = np.array(sorted(cnt)); vs = np.array([cnt[k] for k in ks])
plt.figure(figsize=(11, 3.5)); plt.bar(ks, vs); plt.yscale('log')
plt.xlabel('op-code id'); plt.ylabel('count (log)')
plt.title('Op-code histogram (vocab observed=%d, max id=%d)' % (len(ks), int(ks.max())))
plt.tight_layout(); plt.savefig(FIG / '04_opcode_hist.png', dpi=140); plt.close()
print('saved', FIG / '04_opcode_hist.png', '| vocab observed', len(ks), '| max id', int(ks.max()))

saved C:\Users\user\.vscode\.vscode\NTU\SC4000\kaggle---prj\artifacts\figures\04_opcode_hist.png | vocab observed 69 | max id 118


## 5. node_feat per-column min/max — flag negative & heavy-tailed columns

In [7]:
cmin = np.full(140, np.inf); cmax = np.full(140, -np.inf)
for coll in COLLECTIONS:
    for f in [str(p) for p in list_npz(DATA, coll, 'train')[:25]]:
        with np.load(f) as z:
            nf = z['node_feat'].astype(np.float64)
        cmin = np.minimum(cmin, nf.min(0)); cmax = np.maximum(cmax, nf.max(0))
neg = np.where(cmin < 0)[0]; heavy = np.where((cmin >= 0) & (cmax > 1000))[0]
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(cmax, label='col max'); ax.plot(cmin, label='col min'); ax.set_yscale('symlog')
ax.set_xlabel('node_feat column'); ax.legend()
ax.set_title('node_feat ranges - %d negative cols, %d heavy-tailed (max>1e3)' % (len(neg), len(heavy)))
plt.tight_layout(); plt.savefig(FIG / '05_node_feat_ranges.png', dpi=140); plt.close()
print('negative columns:', list(map(int, neg)))
print('heavy-tailed (log1p) columns:', list(map(int, heavy)))

negative columns: [61, 62, 67]
heavy-tailed (log1p) columns: [21, 22, 23, 24, 27, 28, 29, 43, 44, 107, 108, 109, 110, 111, 112, 117, 118, 119, 120, 122, 123, 124]


## 6. Confirm test labels are placeholders

In [8]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
tr = read_bundle(str(list_npz(DATA, 'tile:xla', 'train')[0]))['config_runtime']
te = read_bundle(str(list_npz(DATA, 'tile:xla', 'test')[0]))['config_runtime']
axes[0].hist(tr, bins=40); axes[0].set_title('TRAIN runtimes (real) min=%d max=%d' % (tr.min(), tr.max()))
axes[1].hist(te, bins=40); axes[1].set_title('TEST runtimes (placeholder) all_zero=%s' % bool((te == 0).all()))
plt.tight_layout(); plt.savefig(FIG / '06_placeholder_check.png', dpi=140); plt.close()
assert bool((te == 0).all()), 'test should be placeholder zeros'
print('confirmed: test runtimes are placeholder zeros; train runtimes are real')

confirmed: test runtimes are placeholder zeros; train runtimes are real


In [9]:
figs = sorted(p.name for p in FIG.glob('0*.png'))
print('Figures rendered:')
for f in figs:
    print(' -', f)
assert len(figs) >= 6, 'expected >=6 figures'
print('EDA notebook complete - gate P1.5 satisfied.')

Figures rendered:
 - 01_counts.png
 - 02_structure_hist.png
 - 03_runtime_ranges.png
 - 04_opcode_hist.png
 - 05_node_feat_ranges.png
 - 06_placeholder_check.png
EDA notebook complete - gate P1.5 satisfied.
